In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/datasprint-2-road-damage-detection/score.py
/kaggle/input/competitions/datasprint-2-road-damage-detection/benchmark_speed.py
/kaggle/input/competitions/datasprint-2-road-damage-detection/inference.py
/kaggle/input/competitions/datasprint-2-road-damage-detection/check_submission.py
/kaggle/input/competitions/datasprint-2-road-damage-detection/test_image_ids.csv
/kaggle/input/competitions/datasprint-2-road-damage-detection/train_baseline.py
/kaggle/input/competitions/datasprint-2-road-damage-detection/val_labels.csv
/kaggle/input/competitions/datasprint-2-road-damage-detection/README.md
/kaggle/input/competitions/datasprint-2-road-damage-detection/val_image_ids.csv
/kaggle/input/competitions/datasprint-2-road-damage-detection/requirements.txt
/kaggle/input/competitions/datasprint-2-road-damage-detection/train_labels.csv
/kaggle/input/competitions/datasprint-2-road-damage-detection/make_kaggle_submission.py
/kaggle/input/competitions/datasprint-2-road-damage-det

In [4]:
import os, json, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm

In [5]:
def find_kit():
    for root, _, fs in os.walk("/kaggle/input"):
        if "train_labels.csv" in fs:
            return Path(root)
    raise FileNotFoundError("Add the competition data to this notebook (Add Input).")


KIT  = find_kit()
WORK = Path("/kaggle/working")
YOLO = WORK / "yolo"                 # normal dataset
YOLO_DEG = WORK / "yolo_degraded"    # degraded val copy (for robustness checks)
SEED = 42
random.seed(SEED); np.random.seed(SEED)
NAMES = ["D00", "D10", "D20", "D40"]
print("KIT:", KIT)

KIT: /kaggle/input/competitions/datasprint-2-road-damage-detection


In [4]:
# converting csv labels to yolo txt 
def link_or_copy(src: Path, dst: Path):
    if dst.exists() or dst.is_symlink():
        return
    try:
        os.symlink(src, dst)          # kit is read-only; symlinks are cheap
    except OSError:
        shutil.copy(src, dst)

def prepare_split(split, labels_csv):
    """images/<split>/*.jpg (symlinks) + labels/<split>/*.txt (one per image, empty if no damage)."""
    img_src = KIT / f"{split}_images"
    img_dst = YOLO / "images" / split
    lab_dst = YOLO / "labels" / split
    img_dst.mkdir(parents=True, exist_ok=True); lab_dst.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(labels_csv) if labels_csv else pd.DataFrame(columns=["image_id","class_id","x_min","y_min","x_max","y_max"])
    df["stem"] = df["image_id"].map(lambda x: Path(str(x)).stem)
    groups = {k: g for k, g in df.groupby("stem")}

    n_boxes = n_skipped = 0
    sizes = {}
    for f in tqdm(sorted(os.listdir(img_src)), desc=f"prepare {split}"):
        stem = Path(f).stem
        link_or_copy(img_src / f, img_dst / f)
        with Image.open(img_src / f) as im:
            W, H = im.size
        sizes[stem] = (W, H)
        lines = []
        if stem in groups:
            for r in groups[stem].itertuples():
                x1 = max(0.0, min(float(r.x_min), W)); x2 = max(0.0, min(float(r.x_max), W))
                y1 = max(0.0, min(float(r.y_min), H)); y2 = max(0.0, min(float(r.y_max), H))
                if x2 - x1 < 1 or y2 - y1 < 1:
                    n_skipped += 1; continue
                cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
                bw, bh = (x2 - x1) / W, (y2 - y1) / H
                lines.append(f"{int(r.class_id)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                n_boxes += 1
        (lab_dst / f"{stem}.txt").write_text("\n".join(lines))   # empty file = background image
    print(f"{split}: {len(sizes)} images, {n_boxes} boxes written, {n_skipped} degenerate skipped")
    return sizes

sizes_train = prepare_split("train", KIT / "train_labels.csv")
sizes_val   = prepare_split("val",   KIT / "val_labels.csv")

DATA_YAML = WORK / "road.yaml"
DATA_YAML.write_text(
    f"path: {YOLO}\ntrain: images/train\nval: images/val\nnames:\n" +
    "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)))
print("wrote", DATA_YAML)

prepare train:   0%|          | 0/14691 [00:00<?, ?it/s]

train: 14691 images, 17517 boxes written, 0 degenerate skipped


prepare val:   0%|          | 0/2150 [00:00<?, ?it/s]

val: 2150 images, 2561 boxes written, 0 degenerate skipped
wrote /kaggle/working/road.yaml


In [5]:
# degrade copy of val to measure robustness
def degrade(img, rng):
    ops = rng.choice(["blur", "dark", "noise", "jpeg"], size=rng.integers(1, 3), replace=False)
    jpeg_q = 95
    for op in ops:
        if op == "blur":
            k = int(rng.choice([3, 5, 7, 9, 11]))
            img = cv2.GaussianBlur(img, (k, k), 0)
        elif op == "dark":
            f = rng.uniform(0.25, 0.7)
            gamma = rng.uniform(1.0, 1.8)
            img = (255 * ((img / 255.0) ** gamma) * f).clip(0, 255).astype(np.uint8)
        elif op == "noise":
            sigma = rng.uniform(6, 28)
            img = (img.astype(np.float32) + rng.normal(0, sigma, img.shape)).clip(0, 255).astype(np.uint8)
        elif op == "jpeg":
            jpeg_q = int(rng.integers(8, 30))
    return img, jpeg_q, list(ops)

def build_degraded_val():
    rng = np.random.default_rng(SEED)
    src_img = YOLO / "images" / "val"; src_lab = YOLO / "labels" / "val"
    dst_img = YOLO_DEG / "images" / "val"; dst_lab = YOLO_DEG / "labels" / "val"
    dst_img.mkdir(parents=True, exist_ok=True)
    if dst_lab.exists() or dst_lab.is_symlink():
        pass
    else:
        (YOLO_DEG / "labels").mkdir(parents=True, exist_ok=True)
        shutil.copytree(src_lab, dst_lab)
    log = {}
    for f in tqdm(sorted(os.listdir(src_img)), desc="degrade val"):
        img = cv2.imread(str(src_img / f))
        out, q, ops = degrade(img, rng)
        cv2.imwrite(str(dst_img / f), out, [cv2.IMWRITE_JPEG_QUALITY, q])
        log[f] = {"ops": ops, "jpeg_q": q}
    (WORK / "degraded_val_log.json").write_text(json.dumps(log))
    (WORK / "road_degraded.yaml").write_text(
        f"path: {YOLO_DEG}\ntrain: images/val\nval: images/val\nnames:\n" +
        "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)))
build_degraded_val()

degrade val:   0%|          | 0/2150 [00:00<?, ?it/s]

In [6]:
# val lists per image size group
def make_size_lists():
    groups = {"600": [], "720": [], "other": []}
    for stem, (W, H) in sizes_val.items():
        key = str(W) if (W == H and str(W) in ("600", "720")) else "other"
        groups[key].append(str(YOLO / "images" / "val" / f"{stem}.jpg"))
    for k, paths in groups.items():
        if not paths:
            continue
        lst = WORK / f"val_{k}.txt"; lst.write_text("\n".join(paths))
        (WORK / f"road_val_{k}.yaml").write_text(
            f"path: {YOLO}\ntrain: images/train\nval: {lst}\nnames:\n" +
            "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)))
        print(k, len(paths), "images")
make_size_lists()

600 1304 images
720 800 images
other 46 images


### yolo model train

In [8]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics] [ultralytics]


In [10]:
# Must be set before the first CUDA allocation (fresh kernel). Reduces OOM from memory fragmentation at 1024 px.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from ultralytics import YOLO as UL


MODEL        = "yolo26s.pt"  
IMGSZ        = 1024
BATCH        = 32            
EPOCHS       = 60        
TIME_HOURS   = None          
PATIENCE     = 15
CLOSE_MOSAIC = 8           
RUN_NAME     = f"{Path(MODEL).stem}_{IMGSZ}"


def build_albu():
    """Extra degradations for robustness. Albumentations arg names differ between v1 and v2, so try both."""
    try:
        import albumentations as A
    except Exception:
        return None
    for make in (
        lambda: [A.GaussianBlur(blur_limit=(3, 9), p=0.15),
                 A.GaussNoise(std_range=(0.02, 0.1), p=0.15),
                 A.ImageCompression(quality_range=(10, 40), p=0.2),
                 A.RandomBrightnessContrast(brightness_limit=(-0.5, 0.15), contrast_limit=0.3, p=0.3),
                 A.RandomGamma(gamma_limit=(80, 200), p=0.15)],
        lambda: [A.GaussianBlur(blur_limit=(3, 9), p=0.15),
                 A.GaussNoise(var_limit=(10, 600), p=0.15),
                 A.ImageCompression(quality_lower=10, quality_upper=40, p=0.2),
                 A.RandomBrightnessContrast(brightness_limit=(-0.5, 0.15), contrast_limit=0.3, p=0.3),
                 A.RandomGamma(gamma_limit=(80, 200), p=0.15)]):
        try:
            return make()
        except Exception:
            continue
    return None


kw = dict(
    data=str(DATA_YAML), imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, device=0,
    seed=SEED, deterministic=True, workers=4, patience=PATIENCE, cos_lr=True,
    close_mosaic=CLOSE_MOSAIC, warmup_epochs=2,
    # geometry: roads have a fixed vertical orientation -> no vertical flip / rotation
    fliplr=0.5, flipud=0.0, degrees=0.0, translate=0.1, scale=0.4, mosaic=1.0, mixup=0.0,
    # colour: strong value jitter for low-light images
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,
    project=str(WORK / "runs"), name=RUN_NAME, exist_ok=True,
    plots=True,
)
albu = build_albu()
if albu:
    kw["augmentations"] = albu

model = UL(MODEL)
try:
    model.train(**kw)
except TypeError:
    kw.pop("augmentations", None)
    model.train(**kw)

# If the session dies mid-run, the weights are in runs/<RUN_NAME>/weights/last.pt and you can continue with:

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[{'__version__': '2.0.8', 'transform': {'__class_fullname__': 'GaussianBlur', 'p': 0.15, 'blur_limit': (3, 9), 'sigma_limit': (0.5, 3.0)}}, {'__version__': '2.0.8', 'transform': {'__class_fullname__': 'GaussNoise', 'p': 0.15, 'mean_range': (0.0, 0.0), 'noise_scale_factor': 1.0, 'per_channel': True, 'std_range': (0.02, 0.1)}}, {'__version__': '2.0.8', 'transform': {'__class_fullname__': 'ImageCompression', 'p': 0.2, 'compression_type': 'jpeg', 'quality_range': (10, 40)}}, {'__version__': '2.0.8', 'transform': {'__class_fullname__': 'RandomBrightnessContrast', 'p': 0.3, 'brightness_by_max': True, 'brightness_limit': (-0.5, 0.15), 'contrast_limit': (-0.3, 0.3), 'ensure_safe_range': False}}, {'__version__': '2.0.8', 'transform': {'__class_fullname__': 'RandomGamma', 'p': 0.15, 'gamma_limit': (80.0, 200.0)}}], auto_augment=

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/60      11.5G      2.129      3.522    0.02118          4       1024: 100% ━━━━━━━━━━━━ 919/919 1.5it/s 10:180.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 68/68 1.6it/s 42.1s0.6ss
                   all       2150       2561      0.188      0.183      0.103     0.0335

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       3/60      11.5G      2.121      3.686    0.02136          3       1024: 100% ━━━━━━━━━━━━ 919/919 1.5it/s 10:120.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 68/68 1.6it/s 41.8s0.6ss
                   all       2150       2561      0.254       0.25      0.147     0.0498

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       4/60      11.4G      2.036      3.579     0.0197          4       1024: 100% ━━━━━━━━━━━━ 919/919 1.5it/s 10:130.3ss
                 Class    

In [15]:
# Evaluate: normal val / degraded val / per size group
BEST = WORK / "runs" / RUN_NAME / "weights" / "best.pt"

def evaluate(yaml_path, tag, imgsz=IMGSZ, iou=0.6):
    m = UL(str(BEST))
    r = m.val(data=str(yaml_path), imgsz=imgsz, conf=0.001, iou=iou, max_det=100,
              batch=32, device=0, plots=False, verbose=False)
    row = {"set": tag, "imgsz": imgsz, "iou": iou, "P": r.box.mp, "R": r.box.mr,
           "mAP50": r.box.map50, "mAP50-95": r.box.map}
    for i, n in enumerate(NAMES):
        if i < len(r.box.ap50):
            row[f"AP50_{n}"] = r.box.ap50[i]
    return row

rows = [evaluate(DATA_YAML, "val_normal"),
        evaluate(WORK / "road_degraded.yaml", "val_degraded")]
for k in ("600", "720"):
    y = WORK / f"road_val_{k}.yaml"
    if y.exists():
        rows.append(evaluate(y, f"val_{k}x{k}"))
res = pd.DataFrame(rows).round(4)
print(res.to_string(index=False))
res.to_csv(WORK / f"eval_summary_{RUN_NAME}.csv", index=False)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO26s summary (fused): 120 layers, 9,466,728 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 189.1±44.0 MB/s, size: 159.0 KB)
val: Scanning /kaggle/working/yolo/labels/val.cache... 2150 images, 896 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2150/2150 409.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 68/68 1.2s/it 1:241.2sss
                   all       2150       2561      0.632      0.536      0.571      0.255
Speed: 1.9ms preprocess, 34.8ms inference, 0.0ms loss, 0.7ms postprocess per image
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO26s summary (fused): 120 layers, 9,466,728 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1651.9±1072.2 MB/s, size: 108.6 KB)
val: Scanning /kaggle/working/yolo_degraded/labels

In [2]:
YOLO_DIR = WORK / "yolo"

In [8]:
FINAL_IMGSZ = IMGSZ      # use the best imgsz / iou found in cell 6
FINAL_IOU   = 0.6
model_final = UL(str(BEST))

def predict_dir(split_dir, imgsz=FINAL_IMGSZ, iou=FINAL_IOU, chunk=32):
    import gc, torch
    gc.collect(); torch.cuda.empty_cache()
    paths = sorted(str(p) for p in Path(split_dir).glob("*.*"))
    out = {}
    for i in tqdm(range(0, len(paths), chunk), desc=f"predict {Path(split_dir).name}"):
        results = model_final.predict(source=paths[i:i + chunk], imgsz=imgsz, conf=0.001, iou=iou,
                                      max_det=100, batch=chunk, device=0, verbose=False)
        for r in results:
            b = r.boxes
            out[Path(r.path).stem] = [(int(c), float(s), *map(float, xy)) for c, s, xy in
                                      zip(b.cls.tolist(), b.conf.tolist(), b.xyxy.tolist())]
    return out

# ---- local scorer (follows the competition text: per-class 101-pt AP, IoU .5:.95; P/R at IoU .5 at the single
#      confidence threshold that maximises mean-over-classes F1; max 100 boxes per image) ----
def _iou_vec(b, G):
    ix1 = np.maximum(b[0], G[:, 0]); iy1 = np.maximum(b[1], G[:, 1])
    ix2 = np.minimum(b[2], G[:, 2]); iy2 = np.minimum(b[3], G[:, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    ua = (b[2] - b[0]) * (b[3] - b[1]) + (G[:, 2] - G[:, 0]) * (G[:, 3] - G[:, 1]) - inter
    return inter / np.maximum(ua, 1e-9)

def _ap101(tp, n_gt):
    if n_gt == 0: return None
    if len(tp) == 0: return 0.0
    tp = np.asarray(tp, float); fp = 1 - tp
    ctp, cfp = np.cumsum(tp), np.cumsum(fp)
    rec = ctp / n_gt; prec = ctp / np.maximum(ctp + cfp, 1e-9)
    prec = np.maximum.accumulate(prec[::-1])[::-1]               # precision envelope
    ap = 0.0
    for r in np.linspace(0, 1, 101):
        idx = np.searchsorted(rec, r, side="left")
        ap += prec[idx] if idx < len(prec) else 0.0
    return ap / 101

def local_score(preds, gt_df, n_classes=4):
    thr_list = np.round(np.arange(0.5, 0.96, 0.05), 2)
    gt = {}
    for r in gt_df.itertuples():
        gt.setdefault((Path(str(r.image_id)).stem, int(r.class_id)), []).append([r.x_min, r.y_min, r.x_max, r.y_max])
    gt = {k: np.array(v, float) for k, v in gt.items()}
    n_gt = {c: sum(len(v) for (i, cc), v in gt.items() if cc == c) for c in range(n_classes)}

    ap_by_thr = {c: [] for c in range(n_classes)}
    pr_inputs = {}
    for c in range(n_classes):
        items = []
        for img, rows in preds.items():
            rows = sorted(rows, key=lambda x: -x[1])[:100]
            for (cc, s, x1, y1, x2, y2) in rows:
                if cc == c: items.append((s, img, np.array([x1, y1, x2, y2])))
        items.sort(key=lambda x: -x[0])
        ious = [(_iou_vec(b, gt[(img, c)]) if (img, c) in gt else None) for _, img, b in items]
        scores = np.array([s for s, _, _ in items])
        for t in thr_list:
            used = {k: np.zeros(len(v), bool) for k, v in gt.items() if k[1] == c}
            tp = []
            for (s, img, b), iv in zip(items, ious):
                hit = 0
                if iv is not None:
                    m = np.where(used[(img, c)], -1, iv)
                    j = int(m.argmax())
                    if m[j] >= t:
                        used[(img, c)][j] = True; hit = 1
                tp.append(hit)
            ap_by_thr[c].append(_ap101(tp, n_gt[c]))
            if t == 0.5: pr_inputs[c] = (scores, np.array(tp))
    valid = [c for c in range(n_classes) if n_gt[c] > 0]
    ap50 = float(np.mean([ap_by_thr[c][0] for c in valid]))
    ap5095 = float(np.mean([np.mean(ap_by_thr[c]) for c in valid]))

    best = (-1, 0, 0, 0)
    for tau in np.unique(np.concatenate([pr_inputs[c][0] for c in valid]))[::-1][::max(1, 1)]:
        Ps, Rs, Fs = [], [], []
        for c in valid:
            s, tp = pr_inputs[c]; keep = s >= tau
            TP = tp[keep].sum(); FP = keep.sum() - TP
            P = TP / max(TP + FP, 1e-9); R = TP / n_gt[c]
            Ps.append(P); Rs.append(R); Fs.append(2 * P * R / max(P + R, 1e-9))
        if np.mean(Fs) > best[0]:
            best = (np.mean(Fs), np.mean(Ps), np.mean(Rs), tau)
    _, P, R, tau = best
    acc = 100 * (0.30 * ap50 + 0.25 * ap5095 + 0.15 * P + 0.15 * R) / 0.85
    return dict(mAP50=ap50, mAP50_95=ap5095, P=P, R=R, conf_thr=float(tau), accuracy_score=acc)

# (a) local val score
val_preds = predict_dir(YOLO_DIR / "images" / "val")
val_gt = pd.read_csv(KIT / "val_labels.csv")
vs = local_score(val_preds, val_gt)
print("\nLOCAL VAL accuracy score (approximation of the leaderboard formula):")
print({k: round(v, 4) for k, v in vs.items()})

# (b) test predictions -> submission.csv
test_ids = pd.read_csv(KIT / "test_image_ids.csv")
id_col = "image_id" if "image_id" in test_ids.columns else test_ids.columns[0]
test_preds = predict_dir(KIT / "test_images")

def pred_string(rows):
    if not rows: return "none"
    rows = sorted(rows, key=lambda x: -x[1])[:100]
    return " ".join(f"{c} {s:.5f} {x1:.1f} {y1:.1f} {x2:.1f} {y2:.1f}" for c, s, x1, y1, x2, y2 in rows)

sub = pd.DataFrame({"image_id": test_ids[id_col],
                    "PredictionString": [pred_string(test_preds.get(Path(str(i)).stem, [])) for i in test_ids[id_col]]})
sub.to_csv(WORK / "submission.csv", index=False)

# sanity checks
n_none = int((sub.PredictionString == "none").sum())
n_boxes = sum(len(v) for v in test_preds.values())
print("\nSUBMISSION CHECKS")
print("rows:", len(sub), "| expected:", len(test_ids), "| null cells:", int(sub.PredictionString.isna().sum()),
      "| empty strings:", int((sub.PredictionString.str.strip() == "").sum()))
print(f"images with no detections ('none'): {n_none} ({100*n_none/len(sub):.1f}%) | avg boxes/image: {n_boxes/len(sub):.1f}")
print("saved:", WORK / "submission.csv")

KIT: /kaggle/input/competitions/datasprint-2-road-damage-detection
weights: /kaggle/working/runs/yolo26s_1024/weights/best.pt


predict val_images:   0%|          | 0/135 [00:00<?, ?it/s]


LOCAL VAL accuracy score (approximation of the leaderboard formula):
{'mAP50': 0.5467, 'mAP50_95': 0.2378, 'P': 0.6048, 'R': 0.5209, 'conf_thr': 0.1976, 'accuracy_score': 46.1558}


predict test_images:   0%|          | 0/263 [00:00<?, ?it/s]


SUBMISSION CHECKS
rows: 4200 | expected: 4200 | null cells: 0 | empty strings: 0
images with no detections ('none'): 386 (9.2%) | avg boxes/image: 22.4
saved: /kaggle/working/submission.csv
OK: 94141 detections for 3814 of 4200 test images (3.2 MB)
note: 386 images have no detections; that is allowed
 


In [10]:
import shutil, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
keep = [
    WORK / "runs",                      # weights, results.csv, plots
    WORK / "submission.csv",
    WORK / "val_pred.csv", WORK / "test_pred.csv",
    WORK / "eda_summary.json",
    WORK / "degraded_val_log.json",
]
keep += list(WORK.glob("eval_summary_*.csv"))
keep += list(WORK.glob("*.yaml"))

with zipfile.ZipFile(WORK / "results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in keep:
        if not p.exists():
            continue
        if p.is_dir():
            for f in p.rglob("*"):
                if f.is_file() and not f.is_symlink():
                    z.write(f, f.relative_to(WORK))
        else:
            z.write(p, p.relative_to(WORK))
print("zip size (MB):", round((WORK / "results.zip").stat().st_size / 1e6, 1))

zip size (MB): 49.1


# inference 

In [20]:
import os, gc, json, time, subprocess, logging
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

WORK = Path("/kaggle/working")

def find_kit():
    for root, _, fs in os.walk("/kaggle/input"):
        if "train_labels.csv" in fs:
            return Path(root)
    raise FileNotFoundError("Add the competition data to this notebook (Add Input).")

In [43]:
MAIN_RUN   = "yolo26s_1024"    
SECOND_RUN = "yolo26s_640"     
BASE_IMGSZ = 1024               # image size the MAIN model was trained at
MAIN_PATH   = None             
SECOND_PATH = None            
RUN_TTA = True                  
PRED_KW = {"end2end": False} 


In [44]:
!pip install ultralytics

In [45]:
def find_weights(run_name):
    """Looks for runs/<run_name>/weights/best.pt in /kaggle/working and in your uploaded results dataset
    (/kaggle/input/...). If the dataset still holds an un-extracted results.zip, it extracts it first."""
    if not run_name:
        return None
    zips = []
    for base in ("/kaggle/working", "/kaggle/input"):
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if not d.endswith("_images") and d != "yolo" and d != "yolo_degraded"]
            if root.replace("\\", "/").endswith(f"{run_name}/weights") and "best.pt" in files:
                return Path(root) / "best.pt"
            zips += [Path(root) / f for f in files if f == "results.zip"]
    for z in zips:                                    # fallback: unzip a results.zip that was not auto-extracted
        import zipfile
        dest = WORK / "restored"
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        hit = list(dest.rglob(f"{run_name}/weights/best.pt"))
        if hit:
            return hit[0]
    return None

WEIGHTS   = find_weights(MAIN_RUN)
WEIGHTS_B = find_weights(SECOND_RUN)
print("main weights  :", WEIGHTS)
print("second weights:", WEIGHTS_B)
if WEIGHTS is None:
    raise FileNotFoundError(f"Could not find runs/{MAIN_RUN}/weights/best.pt. Add your results dataset to this notebook "
                            "(Add Input) or set MAIN_RUN to the right folder name.")

main weights  : /kaggle/input/datasets/shivanshdhakad/results/runs/yolo26s_1024/weights/best.pt
second weights: /kaggle/input/datasets/shivanshdhakad/yolo26-640/runs/yolo26s_640/weights/best.pt


In [46]:
def _iou_vec(b, G):
    ix1 = np.maximum(b[0], G[:, 0]); iy1 = np.maximum(b[1], G[:, 1])
    ix2 = np.minimum(b[2], G[:, 2]); iy2 = np.minimum(b[3], G[:, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    ua = (b[2] - b[0]) * (b[3] - b[1]) + (G[:, 2] - G[:, 0]) * (G[:, 3] - G[:, 1]) - inter
    return inter / np.maximum(ua, 1e-9)

def _ap101(tp, n_gt):
    if n_gt == 0: return None
    if len(tp) == 0: return 0.0
    tp = np.asarray(tp, float); fp = 1 - tp
    ctp, cfp = np.cumsum(tp), np.cumsum(fp)
    rec = ctp / n_gt; prec = ctp / np.maximum(ctp + cfp, 1e-9)
    prec = np.maximum.accumulate(prec[::-1])[::-1]
    ap = 0.0
    for r in np.linspace(0, 1, 101):
        idx = np.searchsorted(rec, r, side="left")
        ap += prec[idx] if idx < len(prec) else 0.0
    return ap / 101

def local_score(preds, gt_df, n_classes=4, return_pr=False):
    """preds: {stem: [(cls, score, x1, y1, x2, y2), ...]}; gt_df: image_id,class_id,x_min,y_min,x_max,y_max"""
    thr_list = np.round(np.arange(0.5, 0.96, 0.05), 2)
    gt = {}
    for r in gt_df.itertuples():
        gt.setdefault((Path(str(r.image_id)).stem, int(r.class_id)), []).append([r.x_min, r.y_min, r.x_max, r.y_max])
    gt = {k: np.array(v, float) for k, v in gt.items()}
    n_gt = {c: sum(len(v) for (i, cc), v in gt.items() if cc == c) for c in range(n_classes)}

    ap_by_thr = {c: [] for c in range(n_classes)}
    pr = {}
    for c in range(n_classes):
        items = []
        for img, rows in preds.items():
            for (cc, s, x1, y1, x2, y2) in sorted(rows, key=lambda x: -x[1])[:100]:
                if cc == c: items.append((s, img, np.array([x1, y1, x2, y2])))
        items.sort(key=lambda x: -x[0])
        ious = [(_iou_vec(b, gt[(img, c)]) if (img, c) in gt else None) for _, img, b in items]
        scores = np.array([s for s, _, _ in items])
        for t in thr_list:
            used = {k: np.zeros(len(v), bool) for k, v in gt.items() if k[1] == c}
            tp = []
            for (s, img, b), iv in zip(items, ious):
                hit = 0
                if iv is not None:
                    m = np.where(used[(img, c)], -1, iv)
                    j = int(m.argmax())
                    if m[j] >= t:
                        used[(img, c)][j] = True; hit = 1
                tp.append(hit)
            ap_by_thr[c].append(_ap101(tp, n_gt[c]))
            if t == 0.5: pr[c] = (scores, np.array(tp))
    valid = [c for c in range(n_classes) if n_gt[c] > 0]
    ap50 = float(np.mean([ap_by_thr[c][0] for c in valid]))
    ap5095 = float(np.mean([np.mean(ap_by_thr[c]) for c in valid]))

    taus = np.unique(np.concatenate([pr[c][0] for c in valid])) if any(len(pr[c][0]) for c in valid) else np.array([0.5])
    Ps, Rs, Fs = [], [], []
    for c in valid:
        s, tp = pr[c]; ctp = np.concatenate([[0], np.cumsum(tp)])
        k = np.searchsorted(-s, -taus, side="right")
        TP = ctp[k]; FP = k - TP
        P = TP / np.maximum(TP + FP, 1e-9); R = TP / n_gt[c]
        Ps.append(P); Rs.append(R); Fs.append(2 * P * R / np.maximum(P + R, 1e-9))
    j = int(np.mean(Fs, axis=0).argmax())
    P, R, tau = float(np.mean(Ps, axis=0)[j]), float(np.mean(Rs, axis=0)[j]), float(taus[j])
    acc = 100 * (0.30 * ap50 + 0.25 * ap5095 + 0.15 * P + 0.15 * R) / 0.85
    out = dict(mAP50=ap50, mAP50_95=ap5095, P=P, R=R, conf_thr=tau, accuracy_score=float(acc))
    if return_pr:
        out["_pr"], out["_ngt"] = pr, n_gt
    return out

In [47]:
def cap_boxes(preds, k):
    return {i: sorted(r, key=lambda x: -x[1])[:k] for i, r in preds.items()}


In [48]:
# ---------- per-class score calibration ----------
# AP does not change when each class's scores are changed monotonically. But precision/recall use ONE confidence
# threshold for all classes. We map each class's best-F1 threshold onto a common value so that single threshold
# is (nearly) optimal for every class at once.
def class_thresholds(pr, n_gt):
    th = {}
    for c, (s, tp) in pr.items():
        if len(s) == 0 or n_gt[c] == 0:
            th[c] = None; continue
        ctp = np.cumsum(tp)
        f1 = 2 * ctp / (np.arange(1, len(s) + 1) + n_gt[c])
        th[c] = float(s[int(f1.argmax())])
    return th

def _map_score(s, t, T):
    return s * T / t if s <= t else T + (s - t) * (1 - T) / (1 - t)     # monotone, maps t -> T, keeps [0,1]

def apply_calibration(preds, th, T):
    out = {}
    for img, rows in preds.items():
        new = []
        for (c, s, x1, y1, x2, y2) in rows:
            t = th.get(c)
            if t is not None:
                t = min(max(t, 0.02), 0.98)
                s = _map_score(s, t, T)
            new.append((c, s, x1, y1, x2, y2))
        out[img] = new
    return out

def fit_calibration(preds, gt_df):
    r = local_score(preds, gt_df, return_pr=True)
    th = class_thresholds(r["_pr"], r["_ngt"])
    vals = [t for t in th.values() if t is not None]
    T = float(np.clip(np.median(vals), 0.05, 0.9))
    return th, T

def crossfit_calibration(preds, gt_df, seed=0):
    """Honest estimate: fit on one half of val, score on the other half (and swap)."""
    stems = sorted(preds); np.random.default_rng(seed).shuffle(stems)
    half = len(stems) // 2
    folds = [set(stems[:half]), set(stems[half:])]
    gstem = gt_df["image_id"].map(lambda x: Path(str(x)).stem)
    base, cal = [], []
    for fit, ev in [(0, 1), (1, 0)]:
        pf = {k: preds[k] for k in folds[fit]}; pe = {k: preds[k] for k in folds[ev]}
        gf = gt_df[gstem.isin(folds[fit])]; ge = gt_df[gstem.isin(folds[ev])]
        th, T = fit_calibration(pf, gf)
        base.append(local_score(pe, ge)["accuracy_score"])
        cal.append(local_score(apply_calibration(pe, th, T), ge)["accuracy_score"])
    return float(np.mean(base)), float(np.mean(cal))

In [49]:
# ---------- weighted box fusion (own minimal implementation, no extra library) ----------
def _iou_one(b, B):
    return _iou_vec(b, B) if len(B) else np.array([])

def wbf(pred_list, weights=None, iou_thr=0.55):
    """Fuse predictions of several models. pred_list: list of {stem: [(cls, score, x1,y1,x2,y2)]}."""
    n_models = len(pred_list)
    weights = weights or [1.0] * n_models
    out = {}
    for img in pred_list[0].keys():
        fused = []
        for c in range(4):
            cand = []
            for m, preds in enumerate(pred_list):
                for (cc, s, x1, y1, x2, y2) in preds.get(img, []):
                    if cc == c: cand.append((s * weights[m], m, np.array([x1, y1, x2, y2], float)))
            cand.sort(key=lambda x: -x[0])
            clusters = []     # each: dict(boxes, scores, fused_box)
            for s, m, b in cand:
                best, best_iou = None, iou_thr
                if clusters:
                    fb = np.array([cl["fb"] for cl in clusters])
                    ious = _iou_vec(b, fb)
                    j = int(ious.argmax())
                    if ious[j] > best_iou: best = j
                if best is None:
                    clusters.append(dict(boxes=[b], scores=[s], fb=b.copy()))
                else:
                    cl = clusters[best]; cl["boxes"].append(b); cl["scores"].append(s)
                    w = np.array(cl["scores"]); cl["fb"] = (np.array(cl["boxes"]) * w[:, None]).sum(0) / w.sum()
            for cl in clusters:
                conf = float(np.mean(cl["scores"])) * min(len(cl["scores"]), n_models) / n_models
                fused.append((c, conf, *map(float, cl["fb"])))
        out[img] = sorted(fused, key=lambda x: -x[1])[:100]
    return out

def speed_score(ms):
    return float(np.clip(1 - np.log10(ms / 25.0), 0, 1))     # 25 ms -> 1.0, 250 ms -> 0.0, log scale between

In [50]:
# %% 1. Load models and data (GPU) ------------------------------------------
import torch
from ultralytics import YOLO as UL
from ultralytics.utils import LOGGER
LOGGER.setLevel(logging.ERROR)          # silences the repeated 'end2end' deprecation warnings

KIT = find_kit()
VAL_DIR, TEST_DIR = KIT / "val_images", KIT / "test_images"
val_gt = pd.read_csv(KIT / "val_labels.csv")
model = UL(str(WEIGHTS))
model_b = UL(str(WEIGHTS_B)) if WEIGHTS_B is not None else None
print("KIT:", KIT, "| main model:", MAIN_RUN, "| second model:", "yes" if model_b else "no")

def predict_dir(mdl, folder, imgsz, iou, chunk=32, extra=None):
    """-> {stem: [(cls, score, x1, y1, x2, y2)]}  (original-image pixels, conf >= 0.001, <= 100 boxes)."""
    gc.collect(); torch.cuda.empty_cache()
    paths = sorted(str(p) for p in Path(folder).glob("*.*"))
    kw = dict(PRED_KW); kw.update(extra or {})
    out = {}
    for i in tqdm(range(0, len(paths), chunk), desc=f"predict {Path(folder).name} {imgsz}px iou={iou}"):
        for r in mdl.predict(source=paths[i:i + chunk], imgsz=imgsz, conf=0.001, iou=iou, max_det=100,
                             batch=chunk, device=0, verbose=False, **kw):
            b = r.boxes
            out[Path(r.path).stem] = [(int(c), float(s), *map(float, xy)) for c, s, xy in
                                      zip(b.cls.tolist(), b.conf.tolist(), b.xyxy.tolist())]
    return out

summary = []      # rows for the final comparison table

# %% 2. NMS IoU sweep at the base image size -------------------------------
iou_runs = {}
for iou in (0.40, 0.45, 0.50, 0.55):
    p = predict_dir(model, VAL_DIR, BASE_IMGSZ, iou)
    iou_runs[iou] = (local_score(p, val_gt), p)
    s = iou_runs[iou][0]
    print(f"iou={iou}: acc={s['accuracy_score']:.2f} mAP50={s['mAP50']:.4f} mAP50-95={s['mAP50_95']:.4f} P={s['P']:.4f} R={s['R']:.4f}")
BEST_IOU = max(iou_runs, key=lambda k: iou_runs[k][0]["accuracy_score"])
print(">>> best NMS IoU:", BEST_IOU)
val_cfg = iou_runs[BEST_IOU][1]; best_score = iou_runs[BEST_IOU][0]["accuracy_score"]; BEST_IMGSZ = BASE_IMGSZ
summary.append(dict(config=f"{BASE_IMGSZ}px, NMS iou {BEST_IOU}", acc=best_score, imgsz=BASE_IMGSZ))

KIT: /kaggle/input/competitions/datasprint-2-road-damage-detection | main model: yolo26s_1024 | second model: yes


predict val_images 640px iou=0.4:   0%|          | 0/68 [00:00<?, ?it/s]

iou=0.4: acc=41.61 mAP50=0.4881 mAP50-95=0.1966 P=0.5500 R=0.5041


predict val_images 640px iou=0.45:   0%|          | 0/68 [00:00<?, ?it/s]

iou=0.45: acc=42.17 mAP50=0.4919 mAP50-95=0.1988 P=0.6081 R=0.4662


predict val_images 640px iou=0.5:   0%|          | 0/68 [00:00<?, ?it/s]

iou=0.5: acc=42.17 mAP50=0.4920 mAP50-95=0.2005 P=0.6036 R=0.4679


predict val_images 640px iou=0.55:   0%|          | 0/68 [00:00<?, ?it/s]

iou=0.55: acc=42.13 mAP50=0.4914 mAP50-95=0.2025 P=0.5974 R=0.4696
>>> best NMS IoU: 0.45


In [51]:
# Image-size sweep at the best IoU -----------------------------------
for sz in (960, 1088):
    p = predict_dir(model, VAL_DIR, sz, BEST_IOU)
    s = local_score(p, val_gt)
    print(f"imgsz={sz}: acc={s['accuracy_score']:.2f} mAP50={s['mAP50']:.4f} mAP50-95={s['mAP50_95']:.4f} P={s['P']:.4f} R={s['R']:.4f}")
    if s["accuracy_score"] > best_score + 0.05:        # require a clear win, not noise
        best_score, BEST_IMGSZ, val_cfg = s["accuracy_score"], sz, p
print(">>> best image size:", BEST_IMGSZ, "| accuracy:", round(best_score, 2))
if BEST_IMGSZ != BASE_IMGSZ:
    summary.append(dict(config=f"{BEST_IMGSZ}px, NMS iou {BEST_IOU}", acc=best_score, imgsz=BEST_IMGSZ))

# Cap on boxes per image (post-processing only) ---------------------
cap_scores = {k: local_score(cap_boxes(val_cfg, k), val_gt)["accuracy_score"] for k in (100, 50, 30)}
print({k: round(v, 3) for k, v in cap_scores.items()})
BEST_CAP = 100 if cap_scores[100] >= max(cap_scores.values()) - 0.02 else max(cap_scores, key=cap_scores.get)
print(">>> cap per image:", BEST_CAP)
val_cfg = cap_boxes(val_cfg, BEST_CAP)

# Per-class score calibration (cross-validated on val) ---------------
base_cv, cal_cv = crossfit_calibration(val_cfg, val_gt)
print(f"2-fold cross-fit: without calibration {base_cv:.2f}  |  with calibration {cal_cv:.2f}  (gain {cal_cv-base_cv:+.2f})")
USE_CAL = (cal_cv - base_cv) > 0.10
print(">>> use calibration:", USE_CAL)
th_all, T_all = fit_calibration(val_cfg, val_gt)
print("per-class best thresholds:", {k: (None if v is None else round(v, 3)) for k, v in th_all.items()}, "| common target T:", round(T_all, 3))
if USE_CAL:
    summary.append(dict(config="+ per-class calibration (cross-fit estimate)", acc=best_score + (cal_cv - base_cv), imgsz=BEST_IMGSZ))


predict val_images 960px iou=0.45:   0%|          | 0/68 [00:00<?, ?it/s]

imgsz=960: acc=45.82 mAP50=0.5396 mAP50-95=0.2314 P=0.6085 R=0.5232


predict val_images 1088px iou=0.45:   0%|          | 0/68 [00:00<?, ?it/s]

imgsz=1088: acc=45.90 mAP50=0.5463 mAP50-95=0.2330 P=0.5897 R=0.5305
>>> best image size: 1088 | accuracy: 45.9
{100: 45.902, 50: 45.904, 30: 45.878}
>>> cap per image: 100
2-fold cross-fit: without calibration 45.99  |  with calibration 45.90  (gain -0.09)
>>> use calibration: False
per-class best thresholds: {0: 0.15, 1: 0.193, 2: 0.258, 3: 0.185} | common target T: 0.189


In [52]:
# Test-time augmentation (TTA) --------------------------------------
tta_ok = False
try:
    p_tta = predict_dir(model, VAL_DIR, BEST_IMGSZ, BEST_IOU, extra={"augment": True})
    s = local_score(cap_boxes(p_tta, BEST_CAP), val_gt)
    print(f"TTA: acc={s['accuracy_score']:.2f} (no TTA: {best_score:.2f}) mAP50={s['mAP50']:.4f} mAP50-95={s['mAP50_95']:.4f}")
    tta_ok = True
    summary.append(dict(config="+ TTA", acc=s["accuracy_score"], imgsz=BEST_IMGSZ, tta=True))
except Exception as e:
    print("TTA not available for this model/version:", repr(e)[:200])

# %% 7. Two-model ensemble (e.g. 1024px model + 640px model) --------------
ens_ok = False
if model_b is not None:
    p_b = predict_dir(model_b, VAL_DIR, 640, BEST_IOU)
    for w in ([1, 1], [2, 1]):
        fused = wbf([val_cfg, p_b], weights=w, iou_thr=0.55)
        s = local_score(fused, val_gt)
        print(f"ensemble weights {w}: acc={s['accuracy_score']:.2f} (single model {best_score:.2f})")
    fused = wbf([val_cfg, p_b], weights=[1, 1], iou_thr=0.55)
    s = local_score(fused, val_gt)
    ens_ok = True
    summary.append(dict(config="ensemble (main + 640px model)", acc=s["accuracy_score"], imgsz=BEST_IMGSZ, ens=True))
else:
    print("No second model found - skipping the ensemble.")

# %% 8. Timing and speed-adjusted comparison -------------------------------
def time_model(mdl, imgsz, n=120, extra=None, warm=10):
    """Median ms per image, one image at a time (like the organisers). Uses Ultralytics' own timers + wall clock."""
    paths = sorted(str(p) for p in VAL_DIR.glob("*.*"))[:n + warm]
    kw = dict(PRED_KW); kw.update(extra or {})
    walls, internal = [], []
    for i, p in enumerate(paths):
        torch.cuda.synchronize(); t0 = time.perf_counter()
        r = mdl.predict(source=p, imgsz=imgsz, conf=0.001, iou=BEST_IOU, max_det=100, device=0, verbose=False, **kw)[0]
        torch.cuda.synchronize(); w = (time.perf_counter() - t0) * 1000
        if i >= warm:
            walls.append(w); internal.append(sum(r.speed.values()))
    return float(np.median(walls)), float(np.median(internal))

t_main = time_model(model, BEST_IMGSZ)
print(f"main model @ {BEST_IMGSZ}px: wall {t_main[0]:.1f} ms | pre+infer+post {t_main[1]:.1f} ms")
t_tta = time_model(model, BEST_IMGSZ, extra={"augment": True}) if tta_ok else None
t_ens = None
if ens_ok:
    t_b = time_model(model_b, 640)
    t_ens = (t_main[0] + t_b[0], t_main[1] + t_b[1])

rows = []
for r in summary:
    if r.get("tta"): t = t_tta[0]
    elif r.get("ens"): t = t_ens[0]
    elif r["imgsz"] == BEST_IMGSZ: t = t_main[0]
    else: t = float("nan")
    sp = speed_score(t) if t == t else float("nan")
    rows.append(dict(config=r["config"], local_acc=round(r["acc"], 2), ms_per_img=round(t, 1),
                     speed_score=round(sp, 3), est_final_no_robust=round(0.85 * r["acc"] + 10 * sp, 2)))
table = pd.DataFrame(rows)
print("\nest_final_no_robust = 0.85*accuracy + 10*speed_score  (robustness 5% not included; compare rows to each other)")
print(table.to_string(index=False))
print("Wall-clock timing here includes image loading; confirm the winner with the kit's benchmark_speed.py.")

predict val_images 1088px iou=0.45:   0%|          | 0/68 [00:00<?, ?it/s]

TTA: acc=47.95 (no TTA: 45.90) mAP50=0.5709 mAP50-95=0.2519


predict val_images 640px iou=0.45:   0%|          | 0/68 [00:00<?, ?it/s]

ensemble weights [1, 1]: acc=48.54 (single model 45.90)
ensemble weights [2, 1]: acc=48.36 (single model 45.90)
main model @ 1088px: wall 41.5 ms | pre+infer+post 37.1 ms

est_final_no_robust = 0.85*accuracy + 10*speed_score  (robustness 5% not included; compare rows to each other)
                       config  local_acc  ms_per_img  speed_score  est_final_no_robust
          640px, NMS iou 0.45      42.17         NaN          NaN                  NaN
         1088px, NMS iou 0.45      45.90        41.5        0.780                46.81
                        + TTA      47.95        93.7        0.426                45.02
ensemble (main + 640px model)      48.54        60.7        0.614                47.40
Wall-clock timing here includes image loading; confirm the winner with the kit's benchmark_speed.py.


In [53]:
# %% 9. FINAL: build the submission with the settings you choose ----------
# Edit CFG after reading the table above. Defaults follow the sweeps.
CFG = dict(imgsz=BEST_IMGSZ, iou=BEST_IOU, cap=BEST_CAP, calibrate=USE_CAL, tta=True, ensemble=True)
print("using:", CFG)

def run_config(folder):
    p = predict_dir(model, folder, CFG["imgsz"], CFG["iou"], extra={"augment": True} if CFG["tta"] else None)
    if CFG["ensemble"]:
        p = wbf([p, predict_dir(model_b, folder, 640, CFG["iou"])], weights=[1, 1], iou_thr=0.55)
    return cap_boxes(p, CFG["cap"])

val_final = run_config(VAL_DIR)
print("val (this exact config, before calibration):", {k: round(v, 4) for k, v in local_score(val_final, val_gt).items()})
test_final = run_config(TEST_DIR)
if CFG["calibrate"]:
    th, T = fit_calibration(val_final, val_gt)            # fit on val, apply to test
    val_cal = apply_calibration(val_final, th, T)
    print("val after calibration (optimistic - fitted on val):", round(local_score(val_cal, val_gt)["accuracy_score"], 2))
    test_final = apply_calibration(test_final, th, T)

test_ids = pd.read_csv(KIT / "test_image_ids.csv")
id_col = "image_id" if "image_id" in test_ids.columns else test_ids.columns[0]
def pred_string(rows):
    if not rows: return "none"
    return " ".join(f"{c} {s:.6f} {x1:.1f} {y1:.1f} {x2:.1f} {y2:.1f}"
                    for c, s, x1, y1, x2, y2 in sorted(rows, key=lambda x: -x[1])[:100])
sub = pd.DataFrame({"image_id": test_ids[id_col],
                    "PredictionString": [pred_string(test_final.get(Path(str(i)).stem, [])) for i in test_ids[id_col]]})
OUT = WORK / "submission_improved.csv"
sub.to_csv(OUT, index=False)
print("saved", OUT, "| rows", len(sub), "| 'none':", int((sub.PredictionString == "none").sum()))

r = subprocess.run(["python", str(KIT / "check_submission.py"), "--pred", str(OUT),
                    "--image-ids", str(KIT / "test_image_ids.csv")], capture_output=True, text=True)
print(r.stdout, r.stderr)

using: {'imgsz': 1088, 'iou': 0.45, 'cap': 100, 'calibrate': False, 'tta': True, 'ensemble': True}


predict val_images 1088px iou=0.45:   0%|          | 0/68 [00:00<?, ?it/s]

predict val_images 640px iou=0.45:   0%|          | 0/68 [00:00<?, ?it/s]

val (this exact config, before calibration): {'mAP50': 0.591, 'mAP50_95': 0.2608, 'P': 0.646, 'R': 0.5362, 'conf_thr': 0.2267, 'accuracy_score': 49.3929}


predict test_images 1088px iou=0.45:   0%|          | 0/132 [00:00<?, ?it/s]

predict test_images 640px iou=0.45:   0%|          | 0/132 [00:00<?, ?it/s]

saved /kaggle/working/submission_improved.csv | rows 4200 | 'none': 163
OK: 101761 detections for 4037 of 4200 test images (3.6 MB)
note: 163 images have no detections; that is allowed
 
